In [187]:
import glob
import json
import os
import re
import zipfile
import queue
import threading

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pdfplumber
import requests
import hnswlib
from html2text import html2text
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

In [2]:
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

In [3]:
us_abbrev = {
    "AK": "Alaska", "AL": "Alabama", "AR": "Arkansas", "AS": "American Samoa", "AZ": "Arizona",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DC": "District of Columbia", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "GU": "Guam", "HI": "Hawaii", "IA": "Iowa",
    "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "KS": "Kansas", "KY": "Kentucky",
    "LA": "Louisiana", "MA": "Massachusetts", "MD": "Maryland", "ME": "Maine", "MI": "Michigan",
    "MN": "Minnesota", "MO": "Missouri", "MP": "Northern Mariana Islands", "MS": "Mississippi", "MT": "Montana",
    "NC": "North Carolina", "ND": "North Dakota", "NE": "Nebraska", "NH": "New Hampshire", "NJ": "New Jersey",
    "NM": "New Mexico", "NV": "Nevada", "NY": "New York", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "PR": "Puerto Rico", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TT": "Trust Territories", "TX": "Texas", "UT": "Utah",
    "VA": "Virginia", "VI": "Virgin Islands", "VT": "Vermont", "WA": "Washington", "WI": "Wisconsin",
    "WV": "West Virginia", "WY": "Wyoming",
}
ca_abbrev = {
    "AB": "Alberta", "BC": "British Columbia", "MB": "Manitoba", "NB": "New Brunswick", "NL": "Newfoundland and Labrador",
    "NS": "Nova Scotia", "NT": "Northwest Territories", "NU": "Nunavut", "ON": "Ontario", "PE": "Prince Edward Island",
    "QC": "Quebec", "SK": "Saskatchewan", "YT": "Yukon",
}
intl_abbrev = {
    "1B": "Armenia", "1H": "Estonia", "1R": "Latvia", "1T": "Marshall Islands", "2A": "Slovenia",
    "2B": "Slovakia", "2M": "Germany", "2N": "Czech Republic", "2Q": "Georgia", "A0": "Alberta, Canada",
    "A1": "British Columbia, Canada", "A5": "Nova Scotia, Canada", "A6": "Ontario, Canada", "A8": "Quebec, Canada", "A9": "Saskatchewan, Canada",
    "C0": "United Arab Emirates", "C1": "Argentina", "C3": "Australia", "C4": "Austria", "C5": "Bahamas",
    "C8": "Barbados", "C9": "Belgium", "D0": "Bermuda", "D5": "Brazil", "D8": "Virgin Islands, British",
    "E9": "Cayman Islands", "F3": "Chile", "F4": "China", "F5": "Taiwan, Province of China", "F8": "Colombia",
    "G2": "Costa Rica", "G7": "Denmark", "H3": "El Salvador", "H9": "Finland", "I0": "France",
    "J3": "Greece", "K3": "Hong Kong", "K5": "Hungary", "K7": "India", "K8": "Indonesia",
    "L2": "Ireland", "L3": "Israel", "L6": "Italy", "M0": "Japan", "M2": "Jordan",
    "M3": "Kenya", "M4": "Korea, Democratic People's Republic of ", "M5": "Korea, Republic of", "N4": "Luxembourg", "N5": "Macau",
    "N8": "Malaysia", "O1": "Malta", "O4": "Mauritius", "O5": "Mexico", "O9": "Monaco",
    "P7": "Netherlands", "PR": "Puerto Rico", "Q1": "Viet Nam", "Q2": "New Zealand", "Q5": "Nigeria",
    "Q8": "Norway", "R1": "Panama", "R5": "Peru", "R6": "Philippines", "R9": "Poland",
    "S1": "Portugal", "S3": "Qatar", "S5": "Romania", "T0": "Saudi Arabia", "T3": "South Africa",
    "U0": "Singapore", "U3": "Spain", "V7": "Sweden", "V8": "Switzerland", "W1": "Thailand",
    "W8": "Turkey", "X0": "United Kingdom", "X3": "Uruguay", "XX": "Unknown", "Y7": "Guernsey",
    "Y8": "Isle of Man", "Y9": "Jersey", "Z4": "Canada (Federal Level)", "Z5": "Montenegro",
}

In [4]:
rows = []

with zipfile.ZipFile(os.path.expanduser(
    "~/Box/dsi-core/11th-hour/idi-corporate-structure/submissions.zip"
)) as zf:
    namelist = list(zf.namelist())
    for filename in tqdm(namelist):
        if filename.startswith("CIK") and filename.endswith(".json"):
            cik = filename[3:-5]

            with zf.open(filename) as file:
                data = json.load(file)

            nyse = []
            nasdaq = []
            cboe = []
            otc = []
            for ticker, exchange in zip(data.get("tickers", []), data.get("exchanges", [])):
                if exchange == "NYSE":
                    nyse.append(ticker)
                elif exchange == "Nasdaq":
                    nasdaq.append(ticker)
                elif exchange == "CBOE":
                    cboe.append(ticker)
                elif exchange == "OTC":
                    otc.append(ticker)

            rows.append({
                "cik": cik,
                "ein": data.get("ein"),
                "lei": data.get("lei"),
                "name": data.get("name"),
                "stateOfIncorporation": data.get("stateOfIncorporation"),
                "addr_street1": data.get("addresses", {}).get("mailing", {}).get("street1"),
                "addr_street2": data.get("addresses", {}).get("mailing", {}).get("street2"),
                "addr_city": data.get("addresses", {}).get("mailing", {}).get("city"),
                "addr_stateOrCountry": data.get("addresses", {}).get("mailing", {}).get("stateOrCountry"),
                "addr_zipCode": data.get("addresses", {}).get("mailing", {}).get("zipCode"),
                "addr_country": data.get("addresses", {}).get("mailing", {}).get("country"),
                "busn_street1": data.get("addresses", {}).get("business", {}).get("street1"),
                "busn_street2": data.get("addresses", {}).get("business", {}).get("street2"),
                "busn_city": data.get("addresses", {}).get("business", {}).get("city"),
                "busn_stateOrCountry": data.get("addresses", {}).get("business", {}).get("stateOrCountry"),
                "busn_zipCode": data.get("addresses", {}).get("business", {}).get("zipCode"),
                "busn_country": data.get("addresses", {}).get("business", {}).get("country"),
                "nyse": ",".join(nyse),
                "nasdaq": ",".join(nasdaq),
                "cboe": ",".join(cboe),
                "otc": ",".join(otc),
            })

submissions = pd.DataFrame(rows)

  0%|          | 0/937121 [00:00<?, ?it/s]

In [54]:
# date, cik, accession, exhibit, subsidiary, place
subsidiaries = pd.read_csv("../subsidiaries-from-exhibits-21-8.csv", dtype=str)
subsidiaries["exhibit"] = subsidiaries["exhibit"].astype(int)

In [87]:
model = SentenceTransformer("Vsevolod/company-names-similarity-sentence-transformer")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [385]:
DROP_PARENS = re.compile(r"\s*\([^)]*\)\s*")
DROP_TOKENS = re.compile(r"[.,-/'#:\\]")
ABBREVIATIONS = {
    "CORPORATION": "CORP",
    "COMPANY": "CO",
    "INCORPORATED": "INC",
    "LIMITED": "LTD",
    "ASSOCIATES": "ASSOC",
    "ASSOCIATION": "ASSOC",
    "TELEVISION": "TV",
    "USA": "US",

    "ACQUISITIONS": "ACQUISITION",
    "COMMUNICATIONS": "COMMUNICATION",
    "DEVELOPMENTS": "DEVELOPMENT",
    "ENTERPRISES": "ENTERPRISE",
    "FINANCES": "FINANCE",
    "FINANCIALS": "FINANCIAL",
    "FOUNDATIONS": "FOUNDATION",
    "FUNDINGS": "FUNDING",
    "FUNDS": "FUND",
    "HOLDINGS": "HOLDING",
    "INDUSTRIES": "INDUSTRY",
    "INVESTMENTS": "INVESTMENT",
    "INVESTORS": "INVESTOR",
    "PARTNERS": "PARTNER",
    "RESOURCES": "RESOURCE",
    "SECURITIES": "SECURITY",
    "SERVICES": "SERVICE",
    "SOLUTIONS": "SOLUTION",
    "SYSTEMS": "SYSTEM",
    "TECHNOLOGIES": "TECHNOLOGY",
    "VENTURES": "VENTURE",
}
def clean_name(name):
    return " ".join([
        ABBREVIATIONS.get(word, word)
        for word in DROP_TOKENS.sub(" ", DROP_PARENS.sub("", name.upper())).replace("&", " AND ").split()
    ]) if isinstance(name, str) else None

submissions_nodup = submissions[["cik", "name"]].copy()
submissions_nodup["name"] = submissions_nodup["name"].apply(clean_name)
submissions_nodup = submissions_nodup.dropna().drop_duplicates("name").reset_index(drop=True)
submissions_nodup["name_set"] = submissions_nodup["name"].apply(lambda name: " ".join(sorted(set(name.split()))))

subsidiary_nodup = subsidiaries[["subsidiary", "place"]].rename(columns={"subsidiary": "name"})
subsidiary_nodup["name"] = subsidiary_nodup["name"].apply(clean_name)
subsidiary_nodup = subsidiary_nodup.dropna().drop_duplicates("name").reset_index(drop=True)
subsidiary_nodup["name_set"] = subsidiary_nodup["name"].apply(lambda name: " ".join(sorted(set(name.split()))))

exact_cik = subsidiary_nodup.merge(submissions_nodup[["name", "cik"]], on="name", how="left").groupby("name").first()
set_cik = subsidiary_nodup.merge(submissions_nodup[["name_set", "cik"]], on="name_set", how="left")[["name", "cik"]].groupby("name").first()

subsidiary_nodup = subsidiary_nodup.set_index("name")
subsidiary_nodup["exact_cik"] = exact_cik["cik"]
subsidiary_nodup["set_cik"] = set_cik["cik"]
subsidiary_nodup = subsidiary_nodup.reset_index("name")

In [ ]:
submission_names = model.encode(submissions_nodup["name"].tolist())

In [ ]:
subsidiary_names = model.encode(subsidiary_nodup["name"].tolist())

In [ ]:
search_index = hnswlib.Index(space="cosine", dim=submission_names.shape[1])
search_index.init_index(max_elements=len(submission_names))
search_index.add_items(submission_names, np.arange(len(submission_names)))

In [ ]:
labels, distances = search_index.knn_query(subsidiary_names, k=3)

In [ ]:
subsidiary_nodup["match1_name"] = submissions_nodup.iloc[labels[:, 0]]["name"].reset_index(drop=True)
subsidiary_nodup["match1_cik"] = submissions_nodup.iloc[labels[:, 0]]["cik"].reset_index(drop=True)
subsidiary_nodup["match1_distance"] = distances[:, 0]

subsidiary_nodup["match2_name"] = submissions_nodup.iloc[labels[:, 1]]["name"].reset_index(drop=True)
subsidiary_nodup["match2_cik"] = submissions_nodup.iloc[labels[:, 1]]["cik"].reset_index(drop=True)
subsidiary_nodup["match2_distance"] = distances[:, 1]

subsidiary_nodup["match3_name"] = submissions_nodup.iloc[labels[:, 2]]["name"].reset_index(drop=True)
subsidiary_nodup["match3_cik"] = submissions_nodup.iloc[labels[:, 2]]["cik"].reset_index(drop=True)
subsidiary_nodup["match3_distance"] = distances[:, 2]

In [ ]:
subsidiary_nodup

In [ ]:
fig, ax = plt.subplots()

ax.hist(subsidiary_nodup["match1_distance"], bins=1000, range=(-1e-5, 1), histtype="step")

ax.set_yscale("log")
ax.set_xlabel("best-match distance")

None

In [ ]:
fig, ax = plt.subplots()

ax.hist(subsidiary_nodup["match1_distance"], bins=1000, range=(-1e-5, 5e-3), histtype="step")

ax.set_yscale("log")
ax.set_xlabel("best-match distance")

None

In [ ]:
fig, ax = plt.subplots()

ax.hist(subsidiary_nodup["match1_distance"], bins=1000, range=(-1e-5, 1e-5), histtype="step")

ax.set_yscale("log")
ax.set_xlabel("best-match distance")

None

In [ ]:
subsidiary_nodup.query("match1_distance < 1e-5").sort_values("match1_distance")[["name", "match1_name"]]

In [ ]:
for _, row in subsidiary_nodup.query("1e-5 < match1_distance").sort_values("match1_distance").iloc[:20].iterrows():
    print(f"{row['match1_distance']}\n{row['name']}\n{row['match1_name']}\n")